## Tour Duration Regression 
Purpose
This notebook rebuilds the distance-based SGD regression model that previously reached a reported holdout R² of about 0.621, then checks how consistently it performs across time using five chronological folds.

We will load the episode data, define the target and input, reproduce the model, run cross-validation, and inspect where prediction errors are largest. Run the notebook from the top down. It does not depend on variables from another notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

DATA_RELATIVE_PATH = Path("data/processed/tour_level/telemetry_flagged_episodes_12h.csv")

# Search the current folder and its parents first. Use the known Desktop
# project location as a fallback if this notebook is opened elsewhere.
data_path = next(
    (parent / DATA_RELATIVE_PATH for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / DATA_RELATIVE_PATH).is_file()),
    None,
)
if data_path is None:
    fallback = Path.home() / "Desktop" / "dt-cargo-vehicle-telemetry-" / DATA_RELATIVE_PATH
    data_path = fallback if fallback.is_file() else None
if data_path is None:
    raise FileNotFoundError(
        f"Could not find {DATA_RELATIVE_PATH}. Update DATA_RELATIVE_PATH to your CSV location."
    )

print("Reading:", data_path)

Reading: /Users/divyshah/Desktop/dt-cargo-vehicle-telemetry-/data/processed/tour_level/telemetry_flagged_episodes_12h.csv


In [2]:
episodes = pd.read_csv(data_path)
episodes["start_time"] = pd.to_datetime(episodes["start_time"], utc=True, errors="coerce")

for column in ["total_distance_km", "elapsed_span_hours"]:
    episodes[column] = pd.to_numeric(episodes[column], errors="coerce")

episodes = episodes.replace([np.inf, -np.inf], np.nan)
episodes = episodes.dropna(
    subset=["start_time", "total_distance_km", "elapsed_span_hours"]
).sort_values("start_time").reset_index(drop=True)

# Convert common CSV representations of true/false to actual booleans.
if episodes["telemetry_flagged"].dtype != bool:
    flag_values = episodes["telemetry_flagged"].astype(str).str.strip().str.lower()
    episodes["telemetry_flagged"] = flag_values.map(
        {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}
    ).fillna(False)

print("Scorable episodes:", len(episodes))
print("Flagged episodes:", int(episodes["telemetry_flagged"].sum()))
print("Date range:", episodes["start_time"].min(), "to", episodes["start_time"].max())

Scorable episodes: 18170
Flagged episodes: 279
Date range: 2021-08-26 13:27:59.533000+00:00 to 2024-02-06 10:36:02.269000+00:00


## Define the prediction task and split by time
The target is elapsed_span_hours. The model uses one input: total_distance_km. We sort first, then use the earliest 80% as the training pool and the latest 20% as a holdout. Telemetry-flagged episodes are excluded from fitting, but remain in the holdout score.

This matches the earlier experiment. Vehicle ID, operation category, and cluster label are not model inputs in this distance-only version.

In [3]:
FEATURES = ["total_distance_km"]
TARGET = "elapsed_span_hours"

cut = int(0.80 * len(episodes))
training_pool = episodes.iloc[:cut].copy()
later_holdout = episodes.iloc[cut:].copy()
eligible_training = training_pool.loc[~training_pool["telemetry_flagged"]].copy()

known_vehicles = set(eligible_training["vehicle_id"])
unknown_vehicle_rows = ~later_holdout["vehicle_id"].isin(known_vehicles)

print("Training episodes after telemetry rule:", len(eligible_training))
print("Holdout episodes:", len(later_holdout))
print("Unseen vehicles in holdout:", later_holdout.loc[unknown_vehicle_rows, "vehicle_id"].nunique())
print("Holdout episodes with unseen vehicles:", int(unknown_vehicle_rows.sum()))

Training episodes after telemetry rule: 14365
Holdout episodes: 3634
Unseen vehicles in holdout: 97
Holdout episodes with unseen vehicles: 3367


## Rebuild the SGD regression model
A degree-2 polynomial lets the model learn a curved relationship with distance (distance and distance squared). We scale the input and polynomial terms so gradient descent can fit reliably. The target is scaled during fitting and predictions are converted back to hours.

The pipeline keeps preprocessing inside each model fit. That matters in cross-validation: each fold learns its scaling from that fold's training rows only.

In [4]:
def build_sgd_model():
    distance_features = Pipeline([
        ("scale_input", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale_terms", StandardScaler()),
    ])

    sgd = SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=0.001,
        learning_rate="adaptive",
        eta0=0.001,
        max_iter=5000,
        tol=1e-4,
        random_state=42,
        average=True,
    )

    return Pipeline([
        ("polynomial_distance", distance_features),
        ("sgd", TransformedTargetRegressor(
            regressor=sgd,
            transformer=StandardScaler(),
        )),
    ])

def regression_scores(actual, predicted):
    return {
        "n": len(actual),
        "R2": r2_score(actual, predicted),
        "MAE_hours": mean_absolute_error(actual, predicted),
        "RMSE_hours": np.sqrt(mean_squared_error(actual, predicted)),
    }

## Reproduce the holdout score
Fit on the eligible first 80%, then score every scorable row in the later 20%. Your R² should be near the previously reported 0.621 if the data file and preprocessing are the same. Small differences can happen if the data file changed.

In [5]:
baseline_model = build_sgd_model()
baseline_model.fit(eligible_training[FEATURES], eligible_training[TARGET])
holdout_prediction = baseline_model.predict(later_holdout[FEATURES])

display(pd.Series(regression_scores(later_holdout[TARGET], holdout_prediction)).round(3))

n             3634.000
R2               0.621
MAE_hours        4.927
RMSE_hours       7.375
dtype: float64

### Run five chronological CV folds
Each validation block contains later episodes than its training block. This is more realistic for future trips than randomly mixing earlier and later episodes. The folds are created only within the first 80%; the later holdout is not used to fit these fold models.

Within each fold, flagged episodes are removed from training only. All rows in the fold's validation block are scored so the result follows the same eligibility rule as the holdout evaluation.

In [6]:
cv = TimeSeriesSplit(n_splits=5)
fold_rows = []
oof_parts = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(training_pool), start=1):
    fold_train = training_pool.iloc[train_idx].copy()
    fold_valid = training_pool.iloc[valid_idx].copy()

    fold_train = fold_train.loc[~fold_train["telemetry_flagged"]]
    fold_model = build_sgd_model()
    fold_model.fit(fold_train[FEATURES], fold_train[TARGET])
    prediction = fold_model.predict(fold_valid[FEATURES])
    actual = fold_valid[TARGET].to_numpy()

    scores = regression_scores(actual, prediction)
    fold_rows.append({
        "fold": fold,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        "validation_start": fold_valid["start_time"].min(),
        "validation_end": fold_valid["start_time"].max(),
        **{name: scores[name] for name in ["R2", "MAE_hours", "RMSE_hours"]},
    })

    fold_output = fold_valid[["start_time", TARGET, "telemetry_flagged"]].copy()
    fold_output["fold"] = fold
    fold_output["prediction_hours"] = prediction
    fold_output["error_hours"] = prediction - actual
    fold_output["absolute_error_hours"] = np.abs(fold_output["error_hours"])
    if "max_internal_gap_h" in fold_valid.columns:
        fold_output["max_internal_gap_h"] = fold_valid["max_internal_gap_h"].to_numpy()
    oof_parts.append(fold_output)

fold_results = pd.DataFrame(fold_rows)
display(fold_results.round(3))

/var/folders/f3/djm8twy54lg9bwn9lkdpg_fh0000gn/T/ipykernel_11285/1897111556.py:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(fold_results.round(3))


,fold,train_rows,validation_rows,validation_start,validation_end,R2,MAE_hours,RMSE_hours
0,1,2423,2422,2021-10-25 05:12:44.809000+00:00,2021-12-01 06:00:00.100000+00:00,0.220,9.115,20.104
1,2,4826,2422,2021-12-01 06:09:00.283000+00:00,2022-01-11 13:32:19.044000+00:00,0.060,12.732,37.724
2,3,7210,2422,2022-01-11 13:36:02.506000+00:00,2022-02-05 07:51:25.519000+00:00,0.182,9.555,16.940
3,4,9600,2422,2022-02-05 08:01:12.970000+00:00,2022-03-11 17:13:58.699000+00:00,0.179,8.688,14.809
4,5,11980,2422,2022-03-11 18:03:00.473000+00:00,2022-06-21 07:14:48.353000+00:00,0.201,6.865,11.062


## Summarize average performance and stability
The fold mean is the average score across time periods. The standard deviation shows how much performance changes from fold to fold. Pooled out-of-fold metrics combine all predictions made for validation rows. A large fold-to-fold spread suggests the model behaves differently over time, even if one fold has a high R².

In [8]:
oof = pd.concat(oof_parts, ignore_index=True)
metrics = ["R2", "MAE_hours", "RMSE_hours"]

print("Mean across folds:")
display(fold_results[metrics].mean().round(3))

print("Fold-to-fold standard deviation:")
display(fold_results[metrics].std().round(3))

print("Pooled out-of-fold scores:")
display(pd.Series(regression_scores(oof[TARGET], oof["prediction_hours"])).round(3))

Mean across folds:


R2             0.168
MAE_hours      9.391
RMSE_hours    20.128
dtype: float64

Fold-to-fold standard deviation:


R2             0.063
MAE_hours      2.129
RMSE_hours    10.372
dtype: float64

Pooled out-of-fold scores:


n             12110.000
R2                0.138
MAE_hours         9.391
RMSE_hours       22.163
dtype: float64

### Find which durations are hardest to predict
Group out-of-fold predictions by actual duration. This tells us whether errors are concentrated among short, medium, or long episodes. These groups are diagnostics only and are not model inputs.

In [9]:
oof["duration_group"] = pd.qcut(oof[TARGET], q=5, duplicates="drop")
duration_errors = oof.groupby("duration_group", observed=True).agg(
    episodes=(TARGET, "size"),
    median_actual_hours=(TARGET, "median"),
    MAE_hours=("absolute_error_hours", "mean"),
    median_absolute_error_hours=("absolute_error_hours", "median"),
    RMSE_hours=("error_hours", lambda values: np.sqrt(np.mean(values ** 2))),
)
display(duration_errors.round(2))

print("Validation rows that were telemetry-flagged:", int(oof["telemetry_flagged"].sum()))
print("Total out-of-fold validation rows:", len(oof))

,episodes,median_actual_hours,MAE_hours,median_absolute_error_hours,RMSE_hours
duration_group,,,,,
"(-0.000829, 0.593]",2422,0.11,6.98,7.44,7.04
"(0.593, 2.369]",2422,1.27,5.95,6.21,6.05
"(2.369, 5.167]",2422,3.58,5.14,5.26,5.43
"(5.167, 11.563]",2422,7.71,4.09,3.53,5.01
"(11.563, 690.449]",2422,23.61,24.80,10.13,48.12


Validation rows that were telemetry-flagged: 168
Total out-of-fold validation rows: 12110


## Check whether episode durations shift over time
Compare duration ranges across the training pool, CV validation folds, and later holdout. Large differences would help explain why the holdout score is much higher than the CV scores.

In [10]:
distribution_rows = []

def add_duration_summary(label, values):
    values = pd.Series(values).dropna()
    distribution_rows.append({
        "period": label,
        "episodes": len(values),
        "median_hours": values.median(),
        "p90_hours": values.quantile(0.90),
        "p99_hours": values.quantile(0.99),
        "max_hours": values.max(),
        "nonpositive_count": int((values <= 0).sum()),
    })

add_duration_summary("Training pool", training_pool[TARGET])
add_duration_summary("Later holdout", later_holdout[TARGET])

for fold, rows in oof.groupby("fold"):
    add_duration_summary(f"CV validation fold {fold}", rows[TARGET])

display(pd.DataFrame(distribution_rows).round(2))

,period,episodes,median_hours,p90_hours,p99_hours,max_hours,nonpositive_count
0,Training pool,14536,3.44,21.86,98.07,690.45,0
1,Later holdout,3634,7.32,12.45,65.14,132.95,2
2,CV validation fold 1,2422,4.23,33.80,104.62,379.40,0
3,CV validation fold 2,2422,4.12,36.65,173.36,690.45,0
4,CV validation fold 3,2422,3.51,23.59,91.37,238.40,0
5,CV validation fold 4,2422,3.86,20.65,79.36,172.65,0
6,CV validation fold 5,2422,2.87,11.91,60.85,323.47,0


## Review extreme-duration episodes
This lists episodes at or above the 99th percentile and episodes with nonpositive duration. Review their timestamps, distance, recorded track hours, and internal gaps before deciding whether any should be excluded under a telemetry-quality rule.

In [11]:
review_columns = [
    "vehicle_id",
    "tour_id",
    "episode_no",
    "start_time",
    "end_time",
    "total_distance_km",
    "recorded_track_hours",
    "max_internal_gap_h",
    "elapsed_span_hours",
    "telemetry_flagged",
]
review_columns = [c for c in review_columns if c in episodes.columns]

p99_duration = episodes[TARGET].quantile(0.99)

episodes_for_review = episodes.loc[
    (episodes[TARGET] >= p99_duration) | (episodes[TARGET] <= 0),
    review_columns,
].sort_values("elapsed_span_hours", ascending=False)

print(f"99th-percentile duration: {p99_duration:.2f} hours")
print("Episodes flagged for review:", len(episodes_for_review))
display(episodes_for_review.head(50))

99th-percentile duration: 91.17 hours
Episodes flagged for review: 184


,vehicle_id,tour_id,episode_no,start_time,end_time,total_distance_km,recorded_track_hours,max_internal_gap_h,elapsed_span_hours,telemetry_flagged
6127,17,2862,0,2021-12-20 13:55:17.077000+00:00,2022-01-18 08:22:12.399000+00:00,159.407,49.050472,6.472883,690.448701,False
7077,39,6916,0,2022-01-07 16:48:21.234000+00:00,2022-02-03 09:01:15.589000+00:00,34.010,55.010761,8.013089,640.215099,False
7080,57,10114,0,2022-01-07 17:42:50.404000+00:00,2022-01-26 09:07:18.004000+00:00,319.580,125.535039,4.160109,447.407667,False
6433,13,2104,0,2021-12-23 22:16:45.836000+00:00,2022-01-11 11:34:24.052000+00:00,23.720,24.141547,10.858606,445.293949,False
6507,16,2591,2,2021-12-27 10:26:33.724000+00:00,2022-01-13 09:11:42.186000+00:00,754.145,106.262761,8.541529,406.752351,False
6219,12,1988,0,2021-12-21 13:24:36.907000+00:00,2022-01-06 11:54:22.078000+00:00,124.464,38.062816,10.016043,382.495881,True
4660,1,39,2,2021-11-28 14:03:23.949000+00:00,2021-12-14 09:27:20.859000+00:00,117.449,163.832095,11.783533,379.399142,False
6389,35,6641,3,2021-12-23 08:42:29.338000+00:00,2022-01-06 17:51:25.564000+00:00,67.753,132.615259,3.870517,345.148952,False
12958,56,10060,2,2022-03-29 22:12:12.078000+00:00,2022-04-12 09:40:32.526000+00:00,51.929,59.517253,10.901734,323.472347,False
7057,7,1299,3,2022-01-07 11:36:25.163000+00:00,2022-01-20 05:25:23.508000+00:00,35.304,58.239949,6.105575,305.816207,False


In [12]:
episodes["recorded_track_hours"] = pd.to_numeric(
    episodes["recorded_track_hours"], errors="coerce"
)

episodes["elapsed_minus_recorded_h"] = (
    episodes["elapsed_span_hours"] - episodes["recorded_track_hours"]
)

episodes["elapsed_to_recorded_ratio"] = (
    episodes["elapsed_span_hours"]
    / episodes["recorded_track_hours"].where(
        episodes["recorded_track_hours"] > 0
    )
)

In [13]:
gap_summary = []

for label, frame in [
    ("Training pool", training_pool),
    ("Later holdout", later_holdout),
]:
    data = episodes.loc[frame.index]

    gap_summary.append({
        "period": label,
        "episodes": len(data),
        "median_elapsed_minus_recorded_h":
            data["elapsed_minus_recorded_h"].median(),
        "p90_elapsed_minus_recorded_h":
            data["elapsed_minus_recorded_h"].quantile(0.90),
        "median_elapsed_to_recorded_ratio":
            data["elapsed_to_recorded_ratio"].median(),
        "ratio_over_2_count":
            int((data["elapsed_to_recorded_ratio"] > 2).sum()),
    })

display(pd.DataFrame(gap_summary).round(2))

,period,episodes,median_elapsed_minus_recorded_h,p90_elapsed_minus_recorded_h,median_elapsed_to_recorded_ratio,ratio_over_2_count
0,Training pool,14536,1.10,15.98,1.73,6336
1,Later holdout,3634,2.87,7.98,1.65,1363


In [14]:
review_columns = [
    "vehicle_id", "tour_id", "episode_no",
    "start_time", "end_time", "total_distance_km",
    "recorded_track_hours", "max_internal_gap_h",
    "elapsed_span_hours", "elapsed_minus_recorded_h",
    "elapsed_to_recorded_ratio", "telemetry_flagged",
]

review_columns = [c for c in review_columns if c in episodes.columns]

largest_gaps = episodes.sort_values(
    "elapsed_minus_recorded_h", ascending=False
)[review_columns]

display(largest_gaps.head(30))

,vehicle_id,tour_id,episode_no,start_time,end_time,total_distance_km,recorded_track_hours,max_internal_gap_h,elapsed_span_hours,elapsed_minus_recorded_h,elapsed_to_recorded_ratio,telemetry_flagged
6127,17,2862,0,2021-12-20 13:55:17.077000+00:00,2022-01-18 08:22:12.399000+00:00,159.407,49.050472,6.472883,690.448701,641.398229,14.076291,False
7077,39,6916,0,2022-01-07 16:48:21.234000+00:00,2022-02-03 09:01:15.589000+00:00,34.010,55.010761,8.013089,640.215099,585.204338,11.637998,False
6433,13,2104,0,2021-12-23 22:16:45.836000+00:00,2022-01-11 11:34:24.052000+00:00,23.720,24.141547,10.858606,445.293949,421.152402,18.445129,False
6219,12,1988,0,2021-12-21 13:24:36.907000+00:00,2022-01-06 11:54:22.078000+00:00,124.464,38.062816,10.016043,382.495881,344.433065,10.049069,True
7080,57,10114,0,2022-01-07 17:42:50.404000+00:00,2022-01-26 09:07:18.004000+00:00,319.580,125.535039,4.160109,447.407667,321.872627,3.564006,False
6507,16,2591,2,2021-12-27 10:26:33.724000+00:00,2022-01-13 09:11:42.186000+00:00,754.145,106.262761,8.541529,406.752351,300.489590,3.827798,False
12958,56,10060,2,2022-03-29 22:12:12.078000+00:00,2022-04-12 09:40:32.526000+00:00,51.929,59.517253,10.901734,323.472347,263.955094,5.434934,False
5944,25,4622,0,2021-12-17 11:15:05.171000+00:00,2021-12-30 02:33:38.600000+00:00,24.825,44.115087,11.636314,303.309286,259.194199,6.875409,False
6402,31,6078,0,2021-12-23 11:43:40.051000+00:00,2022-01-04 08:13:38.907000+00:00,16.019,28.979601,9.115352,284.499682,255.520082,9.817240,False
6411,1,43,1,2021-12-23 13:17:21.206000+00:00,2022-01-04 07:23:40.490000+00:00,50.768,28.458421,7.092781,282.105357,253.646935,9.912895,True


Compare two duration definitions using the same distance-only model and time folds. elapsed_span_hours includes gaps between recorded tracks; recorded_track_hours sums the track durations. This comparison helps choose the target—it does not mean the targets are interchangeable.

In [15]:
comparison_data = episodes.copy()

comparison_data["recorded_track_hours"] = pd.to_numeric(
    comparison_data["recorded_track_hours"], errors="coerce"
)

comparison_data = comparison_data.dropna(
    subset=[
        "start_time",
        "total_distance_km",
        "elapsed_span_hours",
        "recorded_track_hours",
    ]
).sort_values("start_time")

# Keep the same positive-duration episodes for both target comparisons.
comparison_data = comparison_data.loc[
    (comparison_data["elapsed_span_hours"] > 0)
    & (comparison_data["recorded_track_hours"] > 0)
].reset_index(drop=True)

comparison_cut = int(0.80 * len(comparison_data))
comparison_pool = comparison_data.iloc[:comparison_cut].copy()

print("Common positive-duration episodes:", len(comparison_data))
print("Episodes in CV pool:", len(comparison_pool))
print("Flagged episodes in CV pool:", int(comparison_pool["telemetry_flagged"].sum()))

Common positive-duration episodes: 18168
Episodes in CV pool: 14534
Flagged episodes in CV pool: 171


In [16]:
target_names = ["elapsed_span_hours", "recorded_track_hours"]
cv = TimeSeriesSplit(n_splits=5)

comparison_fold_rows = []
comparison_oof_parts = []

for target_name in target_names:
    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(comparison_pool), start=1
    ):
        fold_train = comparison_pool.iloc[train_idx].copy()
        fold_valid = comparison_pool.iloc[valid_idx].copy()

        # Apply the same training eligibility rule as before.
        fold_train = fold_train.loc[~fold_train["telemetry_flagged"]]

        model = build_sgd_model()
        model.fit(fold_train[FEATURES], fold_train[target_name])
        prediction = model.predict(fold_valid[FEATURES])
        actual = fold_valid[target_name].to_numpy()

        scores = regression_scores(actual, prediction)
        comparison_fold_rows.append({
            "target": target_name,
            "fold": fold,
            "train_rows": len(fold_train),
            "validation_rows": len(fold_valid),
            "R2": scores["R2"],
            "MAE_hours": scores["MAE_hours"],
            "RMSE_hours": scores["RMSE_hours"],
        })

        comparison_oof_parts.append(pd.DataFrame({
            "target": target_name,
            "actual_hours": actual,
            "prediction_hours": prediction,
        }))

comparison_folds = pd.DataFrame(comparison_fold_rows)
comparison_oof = pd.concat(comparison_oof_parts, ignore_index=True)

display(comparison_folds.round(3))

,target,fold,train_rows,validation_rows,R2,MAE_hours,RMSE_hours
0,elapsed_span_hours,1,2421,2422,0.217,9.109,20.111
1,elapsed_span_hours,2,4824,2422,0.060,12.740,37.728
2,elapsed_span_hours,3,7208,2422,0.182,9.555,16.938
3,elapsed_span_hours,4,9598,2422,0.180,8.677,14.776
4,elapsed_span_hours,5,11978,2422,0.200,6.885,11.109
5,recorded_track_hours,1,2421,2422,0.564,1.236,4.324
6,recorded_track_hours,2,4824,2422,0.351,1.779,6.280
7,recorded_track_hours,3,7208,2422,0.550,1.333,3.347
8,recorded_track_hours,4,9598,2422,0.560,1.333,3.301
9,recorded_track_hours,5,11978,2422,0.727,0.996,1.897


In [17]:
print("Mean and standard deviation across folds:")
display(
    comparison_folds
    .groupby("target")[["R2", "MAE_hours", "RMSE_hours"]]
    .agg(["mean", "std"])
    .round(3)
)

pooled_rows = []

for target_name, group in comparison_oof.groupby("target"):
    scores = regression_scores(
        group["actual_hours"],
        group["prediction_hours"],
    )
    pooled_rows.append({
        "target": target_name,
        "pooled_R2": scores["R2"],
        "pooled_MAE_hours": scores["MAE_hours"],
        "pooled_RMSE_hours": scores["RMSE_hours"],
    })

display(pd.DataFrame(pooled_rows).round(3))

Mean and standard deviation across folds:


R2        MAE_hours        RMSE_hours        
                       mean    std      mean    std       mean     std
target                                                                
elapsed_span_hours    0.168  0.062     9.393  2.128     20.133  10.367
recorded_track_hours  0.550  0.133     1.336  0.284      3.830   1.620

,target,pooled_R2,pooled_MAE_hours,pooled_RMSE_hours
0,elapsed_span_hours,0.138,9.393,22.165
1,recorded_track_hours,0.501,1.336,4.095


In [18]:
comparison_holdout = comparison_data.iloc[comparison_cut:].copy()

holdout_comparison_rows = []

for target_name in ["elapsed_span_hours", "recorded_track_hours"]:
    fit_rows = comparison_pool.loc[
        ~comparison_pool["telemetry_flagged"]
    ]
    score_rows = comparison_holdout

    model = build_sgd_model()
    model.fit(fit_rows[FEATURES], fit_rows[target_name])
    prediction = model.predict(score_rows[FEATURES])

    holdout_comparison_rows.append({
        "target": target_name,
        **regression_scores(score_rows[target_name], prediction),
    })

display(pd.DataFrame(holdout_comparison_rows).round(3))

,target,n,R2,MAE_hours,RMSE_hours
0,elapsed_span_hours,3634,0.626,4.905,7.303
1,recorded_track_hours,3634,0.846,1.162,1.836


In [19]:
tracks_path = data_path.parents[3] / "data/processed/engineered_tracks.csv"

candidate_fields = [
    "long_haul",
    "fleet_test_id",
    "gross_vehicle_weight",
    "total_mass_with_trailer",
    "axle_class",
]

tracks = pd.read_csv(
    tracks_path,
    usecols=lambda column: column in ["vehicle_id", "tour_id", *candidate_fields],
)

print("Available candidate fields:")
for column in candidate_fields:
    if column in tracks.columns:
        print(f"\n{column}:")
        display(tracks[column].value_counts(dropna=False).head(10))

Available candidate fields:

long_haul:


long_haul
False    135159
True      16439
Name: count, dtype: int64


fleet_test_id:


fleet_test_id
1    40048
4    26860
5    25020
6    24648
3    22933
2    12089
Name: count, dtype: int64


gross_vehicle_weight:


gross_vehicle_weight
18000.0    84934
26000.0    34339
NaN        25021
32000.0     5177
25000.0     2127
Name: count, dtype: int64


total_mass_with_trailer:


total_mass_with_trailer
40000.0    109604
NaN         25021
32000.0     13479
26000.0      3494
Name: count, dtype: int64


axle_class:


axle_class
98.0    76632
42.0    26860
NaN     25021
97.0     8302
41.0     6112
12.0     5177
9.0      3494
Name: count, dtype: int64

In [20]:
tracks_path = data_path.parents[3] / "data/processed/engineered_tracks.csv"

tracks = pd.read_csv(
    tracks_path,
    usecols=["vehicle_id", "tour_id", "long_haul"],
)

def mode_or_missing(values):
    values = values.dropna()
    return values.mode().iloc[0] if not values.empty else np.nan

long_haul_by_tour = (
    tracks.groupby(["vehicle_id", "tour_id"], as_index=False)
    .agg(
        long_haul=("long_haul", mode_or_missing),
        distinct_labels=("long_haul", lambda values: values.dropna().nunique()),
        track_rows=("long_haul", "size"),
    )
)

inconsistent_tours = long_haul_by_tour.loc[
    long_haul_by_tour["distinct_labels"] > 1
]

print("Vehicle/tours:", len(long_haul_by_tour))
print("Tours with inconsistent long_haul labels:", len(inconsistent_tours))
print("Tour-level long_haul counts:")
display(long_haul_by_tour["long_haul"].value_counts(dropna=False))
display(inconsistent_tours.head(20))

Vehicle/tours: 13579
Tours with inconsistent long_haul labels: 1168
Tour-level long_haul counts:


long_haul
False    12954
True       625
Name: count, dtype: int64

,vehicle_id,tour_id,long_haul,distinct_labels,track_rows
4,1,4,True,2,16
8,1,8,False,2,18
10,1,10,True,2,18
12,1,12,True,2,23
14,1,14,False,2,34
16,1,16,True,2,15
18,1,18,True,2,28
20,1,20,True,2,17
22,1,22,False,2,12
24,1,24,False,2,21


In [21]:
tracks_path = data_path.parents[3] / "data/processed/engineered_tracks.csv"

tracks = pd.read_csv(
    tracks_path,
    usecols=[
        "vehicle_id", "tour_id", "track_id",
        "start_time", "stop_time", "long_haul",
    ],
)

tracks["start_time"] = pd.to_datetime(
    tracks["start_time"], utc=True, errors="coerce"
)
tracks["stop_time"] = pd.to_datetime(
    tracks["stop_time"], utc=True, errors="coerce"
)

if tracks["long_haul"].dtype != bool:
    tracks["long_haul"] = (
        tracks["long_haul"].astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )

keys = ["vehicle_id", "tour_id"]
tracks = tracks.dropna(
    subset=keys + ["start_time", "stop_time"]
).sort_values(keys + ["start_time"])

previous_stop = tracks.groupby(keys)["stop_time"].shift()
tracks["gap_from_previous_h"] = (
    tracks["start_time"] - previous_stop
).dt.total_seconds() / 3600

tracks["episode_no"] = (
    tracks.groupby(keys)["gap_from_previous_h"]
    .transform(lambda gaps: gaps.gt(12).cumsum())
    .astype(int)
)

def mode_or_missing(values):
    values = values.dropna()
    return values.mode().iloc[0] if not values.empty else np.nan

episode_labels = (
    tracks.groupby(keys + ["episode_no"], as_index=False)
    .agg(
        track_count=("track_id", "size"),
        distinct_long_haul_labels=("long_haul", lambda s: s.dropna().nunique()),
        long_haul_majority=("long_haul", mode_or_missing),
        long_haul_true_share=("long_haul", lambda s: s.mean()),
    )
)

print("Episodes by number of distinct long_haul labels:")
display(episode_labels["distinct_long_haul_labels"].value_counts().sort_index())

mixed_episodes = episode_labels.loc[
    episode_labels["distinct_long_haul_labels"] > 1
]
print("Mixed-label episodes:", len(mixed_episodes))
display(mixed_episodes.head(20))

Episodes by number of distinct long_haul labels:


distinct_long_haul_labels
1    16588
2     1583
Name: count, dtype: int64

Mixed-label episodes: 1583


,vehicle_id,tour_id,episode_no,track_count,distinct_long_haul_labels,long_haul_majority,long_haul_true_share
21,1,4,0,16,2,True,0.750000
25,1,8,0,18,2,False,0.500000
27,1,10,0,18,2,True,0.833333
29,1,12,0,19,2,True,0.736842
32,1,14,0,34,2,False,0.470588
34,1,16,0,15,2,True,0.666667
36,1,18,0,28,2,True,0.571429
38,1,20,0,17,2,True,0.588235
40,1,22,0,12,2,False,0.416667
42,1,24,0,18,2,False,0.333333


In [22]:
tracks_path = data_path.parents[3] / "data/processed/engineered_tracks.csv"

tracks = pd.read_csv(
    tracks_path,
    usecols=[
        "vehicle_id", "tour_id", "track_id",
        "start_time", "stop_time", "long_haul",
    ],
)

tracks["start_time"] = pd.to_datetime(
    tracks["start_time"], utc=True, errors="coerce"
)
tracks["stop_time"] = pd.to_datetime(
    tracks["stop_time"], utc=True, errors="coerce"
)

if tracks["long_haul"].dtype != bool:
    tracks["long_haul"] = (
        tracks["long_haul"].astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )

keys = ["vehicle_id", "tour_id"]
tracks = tracks.dropna(
    subset=keys + ["start_time", "stop_time"]
).sort_values(keys + ["start_time"])

previous_stop = tracks.groupby(keys)["stop_time"].shift()
tracks["gap_from_previous_h"] = (
    tracks["start_time"] - previous_stop
).dt.total_seconds() / 3600

tracks["episode_no"] = (
    tracks.groupby(keys)["gap_from_previous_h"]
    .transform(lambda gaps: gaps.gt(12).cumsum())
    .astype(int)
)

def mode_or_missing(values):
    values = values.dropna()
    return values.mode().iloc[0] if not values.empty else np.nan

episode_labels = (
    tracks.groupby(keys + ["episode_no"], as_index=False)
    .agg(
        track_count=("track_id", "size"),
        distinct_long_haul_labels=("long_haul", lambda s: s.dropna().nunique()),
        long_haul_majority=("long_haul", mode_or_missing),
        long_haul_true_share=("long_haul", lambda s: s.mean()),
    )
)

print("Episodes by number of distinct long_haul labels:")
display(episode_labels["distinct_long_haul_labels"].value_counts().sort_index())

mixed_episodes = episode_labels.loc[
    episode_labels["distinct_long_haul_labels"] > 1
]
print("Mixed-label episodes:", len(mixed_episodes))
display(mixed_episodes.head(20))

Episodes by number of distinct long_haul labels:


distinct_long_haul_labels
1    16588
2     1583
Name: count, dtype: int64

Mixed-label episodes: 1583


,vehicle_id,tour_id,episode_no,track_count,distinct_long_haul_labels,long_haul_majority,long_haul_true_share
21,1,4,0,16,2,True,0.750000
25,1,8,0,18,2,False,0.500000
27,1,10,0,18,2,True,0.833333
29,1,12,0,19,2,True,0.736842
32,1,14,0,34,2,False,0.470588
34,1,16,0,15,2,True,0.666667
36,1,18,0,28,2,True,0.571429
38,1,20,0,17,2,True,0.588235
40,1,22,0,12,2,False,0.416667
42,1,24,0,18,2,False,0.333333


In [23]:
print(episode_labels.columns.tolist())

display(
    episode_labels["distinct_long_haul_labels"]
    .value_counts(dropna=False)
    .rename_axis("distinct_labels_per_episode")
    .reset_index(name="episode_count")
)

display(mixed_episodes.head(20).reset_index(drop=True))

['vehicle_id', 'tour_id', 'episode_no', 'track_count', 'distinct_long_haul_labels', 'long_haul_majority', 'long_haul_true_share']


,distinct_labels_per_episode,episode_count
0,1,16588
1,2,1583


,vehicle_id,tour_id,episode_no,track_count,distinct_long_haul_labels,long_haul_majority,long_haul_true_share
0,1,4,0,16,2,True,0.750000
1,1,8,0,18,2,False,0.500000
2,1,10,0,18,2,True,0.833333
3,1,12,0,19,2,True,0.736842
4,1,14,0,34,2,False,0.470588
5,1,16,0,15,2,True,0.666667
6,1,18,0,28,2,True,0.571429
7,1,20,0,17,2,True,0.588235
8,1,22,0,12,2,False,0.416667
9,1,24,0,18,2,False,0.333333


In [24]:
label_map = episode_labels.drop(columns=["track_count"])

profile = episodes.merge(
    label_map,
    on=["vehicle_id", "tour_id", "episode_no"],
    how="left",
    validate="one_to_one",
)

profile["long_haul_group"] = np.select(
    [
        profile["long_haul_true_share"].eq(1),
        profile["long_haul_true_share"].eq(0),
        profile["long_haul_true_share"].notna(),
    ],
    ["all_long_haul", "all_non_long_haul", "mixed"],
    default="unlabeled",
)

display(
    profile.groupby("long_haul_group", observed=True)
    .agg(
        episodes=("episode_no", "size"),
        median_distance_km=("total_distance_km", "median"),
        median_recorded_hours=("recorded_track_hours", "median"),
        median_elapsed_hours=("elapsed_span_hours", "median"),
    )
    .round(2)
)

,episodes,median_distance_km,median_recorded_hours,median_elapsed_hours
long_haul_group,,,,
all_long_haul,297,111.15,2.43,6.74
all_non_long_haul,16290,12.91,1.08,3.25
mixed,1583,501.68,8.67,12.67


In [25]:
from sklearn.model_selection import TimeSeriesSplit

TARGET_ACTIVE = "recorded_track_hours"

candidate = profile.dropna(
    subset=["start_time", "total_distance_km", TARGET_ACTIVE]
).sort_values("start_time").reset_index(drop=True)

candidate = candidate.loc[candidate[TARGET_ACTIVE] > 0].copy()

candidate["candidate_group"] = np.where(
    candidate["long_haul_true_share"].fillna(0) > 0,
    "contains_long_haul_track",
    "no_long_haul_track",
)

cv_pool = candidate.iloc[:int(0.80 * len(candidate))].copy()
cv = TimeSeriesSplit(n_splits=5)

rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(cv_pool), start=1):
    fold_train = cv_pool.iloc[train_idx].copy()
    fold_valid = cv_pool.iloc[valid_idx].copy()

    # Exclude telemetry-flagged episodes from model fitting.
    fold_train = fold_train.loc[~fold_train["telemetry_flagged"]]

    # Global distance model.
    global_model = build_sgd_model()
    global_model.fit(fold_train[FEATURES], fold_train[TARGET_ACTIVE])
    global_predictions = global_model.predict(fold_valid[FEATURES])

    for group in ["no_long_haul_track", "contains_long_haul_track"]:
        mask = fold_valid["candidate_group"].eq(group).to_numpy()
        train_group = fold_train.loc[fold_train["candidate_group"].eq(group)]
        valid_group = fold_valid.loc[fold_valid["candidate_group"].eq(group)]

        if len(train_group) < 50 or len(valid_group) < 2:
            continue

        group_model = build_sgd_model()
        group_model.fit(train_group[FEATURES], train_group[TARGET_ACTIVE])
        group_predictions = group_model.predict(valid_group[FEATURES])

        # Compare both models on the same validation episodes.
        for model_name, prediction in [
            ("global", global_predictions[mask]),
            ("group_specific", group_predictions),
        ]:
            scores = regression_scores(valid_group[TARGET_ACTIVE], prediction)
            rows.append({
                "fold": fold,
                "group": group,
                "model": model_name,
                "train_rows": len(train_group) if model_name == "group_specific" else len(fold_train),
                "validation_rows": len(valid_group),
                "R2": scores["R2"],
                "MAE_hours": scores["MAE_hours"],
                "RMSE_hours": scores["RMSE_hours"],
            })

group_cv_results = pd.DataFrame(rows)
display(group_cv_results.round(3))

,fold,group,model,train_rows,validation_rows,R2,MAE_hours,RMSE_hours
0,1,no_long_haul_track,global,2421,2041,0.125,1.066,4.499
1,1,no_long_haul_track,group_specific,2248,2041,0.137,1.021,4.467
2,1,contains_long_haul_track,global,2421,381,0.850,2.148,3.232
3,1,contains_long_haul_track,group_specific,173,381,0.899,1.703,2.645
4,2,no_long_haul_track,global,4824,2155,0.051,1.686,6.243
5,2,no_long_haul_track,group_specific,4280,2155,0.057,1.698,6.224
6,2,contains_long_haul_track,global,4824,267,0.646,2.536,6.569
7,2,contains_long_haul_track,group_specific,544,267,0.639,2.496,6.637
8,3,no_long_haul_track,global,7208,2213,0.169,1.259,3.389
9,3,no_long_haul_track,group_specific,6406,2213,0.161,1.303,3.404


In [26]:
cv = TimeSeriesSplit(n_splits=5)
oof_comparison_parts = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(cv_pool), start=1):
    fold_train = cv_pool.iloc[train_idx].copy()
    fold_valid = cv_pool.iloc[valid_idx].copy()
    fold_train = fold_train.loc[~fold_train["telemetry_flagged"]]

    # One global model
    global_model = build_sgd_model()
    global_model.fit(fold_train[FEATURES], fold_train[TARGET_ACTIVE])
    global_prediction = global_model.predict(fold_valid[FEATURES])

    # A separate model for each provisional group
    group_prediction = np.full(len(fold_valid), np.nan)

    for group in ["no_long_haul_track", "contains_long_haul_track"]:
        train_group = fold_train.loc[fold_train["candidate_group"].eq(group)]
        valid_mask = fold_valid["candidate_group"].eq(group).to_numpy()

        if len(train_group) < 50 or valid_mask.sum() == 0:
            continue

        group_model = build_sgd_model()
        group_model.fit(train_group[FEATURES], train_group[TARGET_ACTIVE])
        group_prediction[valid_mask] = group_model.predict(
            fold_valid.loc[valid_mask, FEATURES]
        )

    oof_comparison_parts.append(pd.DataFrame({
        "fold": fold,
        "candidate_group": fold_valid["candidate_group"].to_numpy(),
        "actual_hours": fold_valid[TARGET_ACTIVE].to_numpy(),
        "global_prediction": global_prediction,
        "group_prediction": group_prediction,
    }))

oof_comparison = pd.concat(oof_comparison_parts, ignore_index=True)

In [27]:
overall_rows = []

for model_name, prediction_column in [
    ("global", "global_prediction"),
    ("group_specific", "group_prediction"),
]:
    scored = oof_comparison.dropna(subset=[prediction_column])
    scores = regression_scores(
        scored["actual_hours"],
        scored[prediction_column],
    )
    overall_rows.append({"model": model_name, **scores})

display(pd.DataFrame(overall_rows).round(3))

,model,n,R2,MAE_hours,RMSE_hours
0,global,12110,0.501,1.336,4.095
1,group_specific,12110,0.452,1.342,4.290


In [28]:
tracks_path = data_path.parents[3] / "data/processed/engineered_tracks.csv"

metadata_fields = [
    "gross_vehicle_weight",
    "total_mass_with_trailer",
    "axle_class",
]

tracks_meta = pd.read_csv(
    tracks_path,
    usecols=[
        "vehicle_id", "tour_id", "start_time", "stop_time",
        *metadata_fields,
    ],
)

tracks_meta["start_time"] = pd.to_datetime(
    tracks_meta["start_time"], utc=True, errors="coerce"
)
tracks_meta["stop_time"] = pd.to_datetime(
    tracks_meta["stop_time"], utc=True, errors="coerce"
)

tracks_meta = tracks_meta.dropna(
    subset=["vehicle_id", "tour_id", "start_time", "stop_time"]
).sort_values(["vehicle_id", "tour_id", "start_time"])

keys = ["vehicle_id", "tour_id"]
previous_stop = tracks_meta.groupby(keys)["stop_time"].shift()

tracks_meta["gap_from_previous_h"] = (
    tracks_meta["start_time"] - previous_stop
).dt.total_seconds() / 3600

tracks_meta["episode_no"] = (
    tracks_meta.groupby(keys)["gap_from_previous_h"]
    .transform(lambda gaps: gaps.gt(12).cumsum())
    .astype(int)
)

def mode_or_missing(values):
    values = values.dropna()
    return values.mode().iloc[0] if not values.empty else np.nan

aggregations = {}
for column in metadata_fields:
    aggregations[f"{column}_mode"] = (column, mode_or_missing)
    aggregations[f"{column}_distinct"] = (
        column, lambda values: values.dropna().nunique()
    )

episode_metadata = (
    tracks_meta.groupby(keys + ["episode_no"], as_index=False)
    .agg(**aggregations)
)

for column in metadata_fields:
    distinct = episode_metadata[f"{column}_distinct"]
    print(
        f"{column}:",
        int((distinct == 1).sum()), "episodes with one value;",
        int((distinct > 1).sum()), "with varying values;",
        int((distinct == 0).sum()), "with no value",
    )

gross_vehicle_weight: 13354 episodes with one value; 0 with varying values; 4817 with no value
total_mass_with_trailer: 13354 episodes with one value; 0 with varying values; 4817 with no value
axle_class: 13354 episodes with one value; 0 with varying values; 4817 with no value


In [29]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

spec_cols = [
    "gross_vehicle_weight_mode",
    "total_mass_with_trailer_mode",
    "axle_class_mode",
]

metadata_for_merge = episode_metadata[
    ["vehicle_id", "tour_id", "episode_no", *spec_cols]
]

candidate_specs = episodes.merge(
    metadata_for_merge,
    on=["vehicle_id", "tour_id", "episode_no"],
    how="left",
    validate="one_to_one",
)

candidate_specs = candidate_specs.dropna(
    subset=["start_time", "total_distance_km", "recorded_track_hours"]
).sort_values("start_time").reset_index(drop=True)

candidate_specs = candidate_specs.loc[
    candidate_specs["recorded_track_hours"] > 0
].reset_index(drop=True)

cv_pool_specs = candidate_specs.iloc[
    :int(0.80 * len(candidate_specs))
].copy()

print("Episodes in CV pool:", len(cv_pool_specs))
print("Missing metadata rows:", int(cv_pool_specs[spec_cols].isna().all(axis=1).sum()))

Episodes in CV pool: 14534
Missing metadata rows: 4585


In [30]:
def build_sgd_model_with_specs():
    distance_pipeline = Pipeline([
        ("scale_input", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale_terms", StandardScaler()),
    ])

    specs_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent", add_indicator=True)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessing = ColumnTransformer([
        ("distance", distance_pipeline, ["total_distance_km"]),
        ("specs", specs_pipeline, spec_cols),
    ])

    sgd = SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=0.001,
        learning_rate="adaptive",
        eta0=0.001,
        max_iter=5000,
        tol=1e-4,
        random_state=42,
        average=True,
    )

    return Pipeline([
        ("prep", preprocessing),
        ("sgd", TransformedTargetRegressor(
            regressor=sgd,
            transformer=StandardScaler(),
        )),
    ])


cv = TimeSeriesSplit(n_splits=5)
fold_rows = []
oof_rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(cv_pool_specs), start=1):
    fold_train = cv_pool_specs.iloc[train_idx].copy()
    fold_valid = cv_pool_specs.iloc[valid_idx].copy()

    fold_train = fold_train.loc[~fold_train["telemetry_flagged"]]

    distance_model = build_sgd_model()
    distance_model.fit(fold_train[FEATURES], fold_train["recorded_track_hours"])
    distance_pred = distance_model.predict(fold_valid[FEATURES])

    specs_model = build_sgd_model_with_specs()
    specs_model.fit(fold_train[["total_distance_km", *spec_cols]],
                    fold_train["recorded_track_hours"])
    specs_pred = specs_model.predict(
        fold_valid[["total_distance_km", *spec_cols]]
    )

    for name, prediction in [
        ("distance_only", distance_pred),
        ("distance_plus_specs", specs_pred),
    ]:
        scores = regression_scores(
            fold_valid["recorded_track_hours"], prediction
        )
        fold_rows.append({"fold": fold, "model": name, **scores})

    oof_rows.append(pd.DataFrame({
        "actual_hours": fold_valid["recorded_track_hours"].to_numpy(),
        "distance_only": distance_pred,
        "distance_plus_specs": specs_pred,
    }))

display(pd.DataFrame(fold_rows).round(3))

oof_specs = pd.concat(oof_rows, ignore_index=True)
pooled_rows = []

for name in ["distance_only", "distance_plus_specs"]:
    scores = regression_scores(oof_specs["actual_hours"], oof_specs[name])
    pooled_rows.append({"model": name, **scores})

display(pd.DataFrame(pooled_rows).round(3))

,fold,model,n,R2,MAE_hours,RMSE_hours
0,1,distance_only,2422,0.564,1.236,4.324
1,1,distance_plus_specs,2422,0.566,1.226,4.316
2,2,distance_only,2422,0.351,1.779,6.280
3,2,distance_plus_specs,2422,0.356,1.722,6.260
4,3,distance_only,2422,0.550,1.333,3.347
5,3,distance_plus_specs,2422,0.556,1.231,3.324
6,4,distance_only,2422,0.560,1.333,3.301
7,4,distance_plus_specs,2422,0.573,1.180,3.252
8,5,distance_only,2422,0.727,0.996,1.897
9,5,distance_plus_specs,2422,0.747,0.834,1.824


,model,n,R2,MAE_hours,RMSE_hours
0,distance_only,12110,0.501,1.336,4.095
1,distance_plus_specs,12110,0.508,1.238,4.068


In [31]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

feature_sets = {
    "distance_only": [],
    "distance_plus_gross_weight": ["gross_vehicle_weight_mode"],
    "distance_plus_total_mass": ["total_mass_with_trailer_mode"],
    "distance_plus_axle_class": ["axle_class_mode"],
    "distance_plus_all_specs": spec_cols,
}

def build_model_for_features(metadata_columns):
    distance_pipeline = Pipeline([
        ("scale_input", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale_terms", StandardScaler()),
    ])

    sgd = SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=0.001,
        learning_rate="adaptive",
        eta0=0.001,
        max_iter=5000,
        tol=1e-4,
        random_state=42,
        average=True,
    )

    target_model = TransformedTargetRegressor(
        regressor=sgd,
        transformer=StandardScaler(),
    )

    if not metadata_columns:
        return Pipeline([
            ("polynomial_distance", distance_pipeline),
            ("sgd", target_model),
        ])

    metadata_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent", add_indicator=True)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessing = ColumnTransformer([
        ("distance", distance_pipeline, ["total_distance_km"]),
        ("metadata", metadata_pipeline, metadata_columns),
    ])

    return Pipeline([
        ("prep", preprocessing),
        ("sgd", target_model),
    ])

cv = TimeSeriesSplit(n_splits=5)
ablation_rows = []
oof_actual = []
oof_predictions = {name: [] for name in feature_sets}

for fold, (train_idx, valid_idx) in enumerate(cv.split(cv_pool_specs), start=1):
    fold_train = cv_pool_specs.iloc[train_idx].copy()
    fold_valid = cv_pool_specs.iloc[valid_idx].copy()
    fold_train = fold_train.loc[~fold_train["telemetry_flagged"]]

    y_valid = fold_valid["recorded_track_hours"].to_numpy()
    oof_actual.extend(y_valid)

    for model_name, metadata_columns in feature_sets.items():
        input_columns = ["total_distance_km", *metadata_columns]
        model = build_model_for_features(metadata_columns)

        model.fit(fold_train[input_columns], fold_train["recorded_track_hours"])
        prediction = model.predict(fold_valid[input_columns])
        oof_predictions[model_name].extend(prediction)

        scores = regression_scores(y_valid, prediction)
        ablation_rows.append({
            "fold": fold,
            "model": model_name,
            "R2": scores["R2"],
            "MAE_hours": scores["MAE_hours"],
            "RMSE_hours": scores["RMSE_hours"],
        })

display(
    pd.DataFrame(ablation_rows)
    .groupby("model")[["R2", "MAE_hours", "RMSE_hours"]]
    .agg(["mean", "std"])
    .round(3)
)

pooled_rows = []
for model_name, predictions in oof_predictions.items():
    scores = regression_scores(oof_actual, predictions)
    pooled_rows.append({"model": model_name, **scores})

display(pd.DataFrame(pooled_rows).round(3))

R2        MAE_hours        RMSE_hours       
                             mean    std      mean    std       mean    std
model                                                                      
distance_only               0.550  0.133     1.336  0.284      3.830  1.620
distance_plus_all_specs     0.560  0.139     1.238  0.317      3.795  1.639
distance_plus_axle_class    0.560  0.139     1.242  0.317      3.795  1.639
distance_plus_gross_weight  0.556  0.136     1.262  0.321      3.808  1.630
distance_plus_total_mass    0.558  0.138     1.247  0.312      3.800  1.638

,model,n,R2,MAE_hours,RMSE_hours
0,distance_only,12110,0.501,1.336,4.095
1,distance_plus_gross_weight,12110,0.505,1.262,4.077
2,distance_plus_total_mass,12110,0.506,1.247,4.073
3,distance_plus_axle_class,12110,0.508,1.242,4.069
4,distance_plus_all_specs,12110,0.508,1.238,4.068


In [32]:
split = int(0.80 * len(candidate_specs))
train_pool = candidate_specs.iloc[:split].copy()
later_holdout = candidate_specs.iloc[split:].copy()
train_pool = train_pool.loc[~train_pool["telemetry_flagged"]]

holdout_rows = []

for name, metadata_columns in [
    ("distance_only", []),
    ("distance_plus_axle", ["axle_class_mode"]),
]:
    input_columns = ["total_distance_km", *metadata_columns]
    model = build_model_for_features(metadata_columns)

    model.fit(train_pool[input_columns], train_pool["recorded_track_hours"])
    prediction = model.predict(later_holdout[input_columns])

    holdout_rows.append({
        "model": name,
        **regression_scores(
            later_holdout["recorded_track_hours"],
            prediction,
        ),
    })

display(pd.DataFrame(holdout_rows).round(3))

,model,n,R2,MAE_hours,RMSE_hours
0,distance_only,3634,0.846,1.162,1.836
1,distance_plus_axle,3634,0.850,1.170,1.809


In [33]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "anomaly_detection/reports/quality_v2").is_dir()
)
REPORTS = ROOT / "anomaly_detection/reports/quality_v2"

core = pd.read_csv(
    REPORTS / "validation_predictions_core_four_feature.csv"
)
extended = pd.read_csv(
    REPORTS / "validation_predictions_extended_six_feature.csv"
)

validation_summary = json.loads(
    (REPORTS / "validation_summary.json").read_text()
)

In [34]:
core_scores = core[
    ["track_id", "anomaly_score", "is_anomaly", "quality_status_core"]
].rename(columns={
    "anomaly_score": "core_score",
    "is_anomaly": "core_alert",
})

extended_scores = extended[
    ["track_id", "anomaly_score", "is_anomaly", "quality_status_extended"]
].rename(columns={
    "anomaly_score": "extended_score",
    "is_anomaly": "extended_alert",
})

ensemble = core_scores.merge(
    extended_scores,
    on="track_id",
    validate="one_to_one",
)

ensemble["consensus_alert"] = (
    ensemble["core_alert"] & ensemble["extended_alert"]
)
ensemble["union_alert"] = (
    ensemble["core_alert"] | ensemble["extended_alert"]
)

# Normalize the two score scales before averaging them.
ensemble["core_percentile"] = ensemble["core_score"].rank(pct=True)
ensemble["extended_percentile"] = ensemble["extended_score"].rank(pct=True)
ensemble["rank_average"] = (
    ensemble["core_percentile"] + ensemble["extended_percentile"]
) / 2

# Diagnostic comparison: choose the top 2% of this validation set.
rank_cutoff = ensemble["rank_average"].quantile(0.98)
ensemble["rank_average_alert"] = ensemble["rank_average"] >= rank_cutoff

In [35]:
quality_warning = (
    ensemble["quality_status_core"].eq("eligible_with_warnings")
    | ensemble["quality_status_extended"].eq("eligible_with_warnings")
)

rules = {
    "core model": ensemble["core_alert"],
    "extended model": ensemble["extended_alert"],
    "consensus": ensemble["consensus_alert"],
    "union": ensemble["union_alert"],
    "rank average": ensemble["rank_average_alert"],
}

comparison = pd.DataFrame([
    {
        "rule": name,
        "alerts": int(mask.sum()),
        "alert_rate_pct": round(mask.mean() * 100, 2),
        "alerts_with_quality_warnings": int((mask & quality_warning).sum()),
    }
    for name, mask in rules.items()
])

comparison

,rule,alerts,alert_rate_pct,alerts_with_quality_warnings
0,core model,607,2.0,31
1,extended model,607,2.0,36
2,consensus,425,1.4,16
3,union,789,2.6,51
4,rank average,607,2.0,27


## Optimized Duration Estimator: HistGradientBoosting with Monotonic Distance & Log-Target Transform

To overcome the limitations of the polynomial SGD baseline (extreme upward bias on short trips and numerical instability), we introduce an optimized pipeline:
- **Algorithm**: Histogram-based Gradient Boosting ()
- **Features**:  paired with generalizable fleet specification  from  (avoiding unseen vehicle ID overfitting)
- **Monotonic Constraint**: Distance must have a non-negative relationship with duration ()
- **Target Transformation**: $\log(1 + y)$ target scaling to stabilize the heavy-tailed duration distribution and reduce large residual penalties


In [ ]:
# Fit and evaluate the optimized HistGradientBoosting pipeline on the chronological holdout
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# Merge fleet axle_class
fleet_meta = pd.read_csv(data_path.parents[3] / "data/raw/fleets/fleet.csv")
episodes_enriched = episodes.merge(fleet_meta[["vehicle_id", "axle_class"]], on="vehicle_id", how="left")

train_enriched = episodes_enriched.iloc[:cut].copy()
holdout_enriched = episodes_enriched.iloc[cut:].copy()
train_eligible_enriched = train_enriched.loc[~train_enriched["telemetry_flagged"]].copy()

features_opt = ["total_distance_km", "axle_class"]

optimized_model = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=150,
        min_samples_leaf=40,
        l2_regularization=2.0,
        categorical_features=["axle_class"],
        monotonic_cst=[1, 0],
        random_state=42,
    ),
    func=np.log1p,
    inverse_func=np.expm1,
)

optimized_model.fit(train_eligible_enriched[features_opt], train_eligible_enriched[TARGET])
opt_holdout_pred = np.maximum(0.0, optimized_model.predict(holdout_enriched[features_opt]))

holdout_comparison_df = pd.DataFrame([
    {"Model": "Baseline SGD (Distance Poly-2)", **regression_scores(later_holdout[TARGET], holdout_prediction)},
    {"Model": "Optimized HGB (Dist + Axle + Log1p)", **regression_scores(holdout_enriched[TARGET], opt_holdout_pred)},
])
display(holdout_comparison_df.round(3))


In [ ]:
# 5-Fold Chronological TimeSeriesSplit Cross-Validation Comparison
cv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(training_pool), start=1):
    # SGD baseline fold
    f_train_sgd = training_pool.iloc[train_idx].loc[lambda df: ~df["telemetry_flagged"]]
    f_valid_sgd = training_pool.iloc[valid_idx]
    m_sgd = build_sgd_model()
    m_sgd.fit(f_train_sgd[FEATURES], f_train_sgd[TARGET])
    p_sgd = np.maximum(0.0, m_sgd.predict(f_valid_sgd[FEATURES]))
    cv_rows.append({"fold": fold, "model": "Baseline SGD", **regression_scores(f_valid_sgd[TARGET], p_sgd)})

    # Optimized HGB fold
    f_train_hgb = train_enriched.iloc[train_idx].loc[lambda df: ~df["telemetry_flagged"]]
    f_valid_hgb = train_enriched.iloc[valid_idx]
    m_hgb = TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=150,
            min_samples_leaf=40,
            l2_regularization=2.0,
            categorical_features=["axle_class"],
            monotonic_cst=[1, 0],
            random_state=42,
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    )
    m_hgb.fit(f_train_hgb[features_opt], f_train_hgb[TARGET])
    p_hgb = np.maximum(0.0, m_hgb.predict(f_valid_hgb[features_opt]))
    cv_rows.append({"fold": fold, "model": "Optimized HGB", **regression_scores(f_valid_hgb[TARGET], p_hgb)})

df_cv_comparison = pd.DataFrame(cv_rows)
print("Mean across folds:")
display(df_cv_comparison.groupby("model")[["R2", "MAE_hours", "RMSE_hours"]].mean().round(3))
print("Fold-to-fold standard deviation:")
display(df_cv_comparison.groupby("model")[["R2", "MAE_hours", "RMSE_hours"]].std().round(3))
